### Scott 10K clustering

#### For (AD, MCI and CN) and (AD,CN):
#### 1. Elbow plot - ideal number of clusters in theory?
#### 2. (with both UMAP, t-SNE and PCA) Regular K-means cluster based on k=3 or k=2 based on how many diagnoses involved. INCLUDE SILHOUETTE SCORE!!!
#### 3. Compare clusters with gold-standard diagnoses values
#### 4. Colour with confounders (NB: for some confounders, figure legends need to be outside of the graph but this means they get clipped out - to fix in image editor):

1. Age
2. Gender
3. Scanner field strength
4. APOE4 genotype
5. MMSE total score
6. CDR total score
7. NPI-Q score
8. Amyloid beta levels
9. Tau blood levels
10. Beta coefficient values

#### Clustering time!!!

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from matplotlib.colors import ListedColormap
from matplotlib import font_manager

font_path = "C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/arial.ttf"
font_manager.fontManager.addfont(font_path)


confounders = ['PTAGE']

column_dicts = {
    'desikan_coeffs': ['desikan_D1', 'desikan_D2', 'desikan_DAT', 'desikan_NET', 'desikan_5HT1A',
                       'desikan_5HT1B', 'desikan_5HT2A', 'desikan_5HT4', 'desikan_5HT6', 'desikan_5HTT',
                       'desikan_a4b2', 'desikan_M1', 'desikan_vAChT', 'desikan_NMDA', 'desikan_mGluR5',
                       'desikan_GABAA/BZ', 'desikan_H3', 'desikan_CB1', 'desikan_MOR'],
    'flipped_desikan_coeffs': ['flipped_desikan_D1', 'flipped_desikan_D2', 'flipped_desikan_DAT',
                               'flipped_desikan_NET', 'flipped_desikan_5HT1A', 'flipped_desikan_5HT1B',
                               'flipped_desikan_5HT2A', 'flipped_desikan_5HT4', 'flipped_desikan_5HT6',
                               'flipped_desikan_5HTT', 'flipped_desikan_a4b2', 'flipped_desikan_M1',
                               'flipped_desikan_vAChT', 'flipped_desikan_NMDA', 'flipped_desikan_mGluR5',
                               'flipped_desikan_GABAA/BZ', 'flipped_desikan_H3', 'flipped_desikan_CB1',
                               'flipped_desikan_MOR'],
    'destrieux_coeffs': ['destrieux_D1', 'destrieux_D2', 'destrieux_DAT', 'destrieux_NET',
                         'destrieux_5HT1A', 'destrieux_5HT1B', 'destrieux_5HT2A', 'destrieux_5HT4',
                         'destrieux_5HT6', 'destrieux_5HTT', 'destrieux_a4b2', 'destrieux_M1',
                         'destrieux_vAChT', 'destrieux_NMDA', 'destrieux_mGluR5', 'destrieux_GABAA/BZ',
                         'destrieux_H3', 'destrieux_CB1', 'destrieux_MOR'],
    'flipped_destrieux_coeffs': ['flipped_destrieux_D1', 'flipped_destrieux_D2', 'flipped_destrieux_DAT',
                                 'flipped_destrieux_NET', 'flipped_destrieux_5HT1A', 'flipped_destrieux_5HT1B',
                                 'flipped_destrieux_5HT2A', 'flipped_destrieux_5HT4', 'flipped_destrieux_5HT6',
                                 'flipped_destrieux_5HTT', 'flipped_destrieux_a4b2', 'flipped_destrieux_M1',
                                 'flipped_destrieux_vAChT', 'flipped_destrieux_NMDA', 'flipped_destrieux_mGluR5',
                                 'flipped_destrieux_GABAA/BZ', 'flipped_destrieux_H3', 'flipped_destrieux_CB1',
                                 'flipped_destrieux_MOR']
}



df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv',
                 low_memory=False)

three_diag_df = df[df['DIAGNOSIS'].isin([1, 2, 3])].copy()
two_diag_df   = df[df['DIAGNOSIS'].isin([1, 3])].copy()

scaler = StandardScaler()
seed   = 42

color_map      = {1: 'green', 2: 'blue', 3: 'red'}
cluster_colors = {0: 'cyan',  1: 'magenta', 2: 'indigo'}

cmap_data = np.genfromtxt('C:/Users/User/Downloads/deconstructalz.github.io/standards/colourmap.csv',
                          delimiter=',')
cmap_div = ListedColormap(cmap_data)


for cohort in [three_diag_df, two_diag_df]:
    diag_vals = sorted(cohort['DIAGNOSIS'].unique())
    k = len(diag_vals)
    filepath_specifics = 'three_diags' if k == 3 else 'two_diags'
    title_specifics = "CN, MCI and AD" if k == 3 else "CN, AD"
    base_dir = f'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_analysis/clusters/{filepath_specifics}'

    for col_key, feature_cols in column_dicts.items():
        
        print(confounders)
        
        print(f'{filepath_specifics} - {col_key}...')

        # 1. Prepare data

        working_cols = feature_cols + ['DIAGNOSIS'] + confounders
        data_subset  = cohort[working_cols].dropna(subset=feature_cols).reset_index(drop=True)

        y_true     = data_subset['DIAGNOSIS']
        X_unscaled = data_subset[feature_cols].copy()

        # 2. Scale
        X_scaled    = scaler.fit_transform(X_unscaled)
        X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols) 

        # 3. Cluster
        kmeans         = KMeans(n_clusters=k, random_state=seed)
        cluster_labels = kmeans.fit_predict(X_scaled) 
        sil_score      = silhouette_score(X_scaled, cluster_labels)
        print(f'Silhouette score: {sil_score:.2f}')

        # 4. Attach cluster labels by position

        X_unscaled['Clusters']  = cluster_labels
        X_scaled_df['Clusters'] = cluster_labels

        # 5. Dimensionality reduction
        print('PCA...')
        pca_res  = PCA(n_components=2).fit_transform(X_scaled)
        print('TSNE...')
        tsne_res = TSNE(n_components=2, random_state=seed).fit_transform(X_scaled)
        print('UMAP...')
        umap_res = umap.UMAP(random_state=seed).fit_transform(X_scaled)

        # 6. Build visualisation dataframe
        viz_df = pd.DataFrame({
            'Clusters':  cluster_labels,
            'DIAGNOSES': y_true.values,
            'PCA 1':     pca_res[:, 0],  'PCA 2':  pca_res[:, 1],
            'TSNE 1':    tsne_res[:, 0], 'TSNE 2': tsne_res[:, 1],
            'UMAP 1':    umap_res[:, 0], 'UMAP 2': umap_res[:, 1],
        })
        for conf in [c for c in working_cols if c != 'DIAGNOSIS']:
            viz_df[conf] = data_subset[conf].values


        # 8. Scatter plots
        dims = [('PCA 1', 'PCA 2'), ('TSNE 1', 'TSNE 2'), ('UMAP 1', 'UMAP 2')]
        for d1, d2 in dims:
            method    = d1.split(' ')[0]
            base_path = f'{base_dir}/{method}_k{k}_{col_key}'


            # C. Colour by confounder
            for conf in [c for c in working_cols if c != 'DIAGNOSIS']:
                plt.figure(figsize=(8, 6))
                if viz_df[conf].dtype == 'object' or viz_df[conf].nunique() < 10:
                    sns.scatterplot(data=viz_df, x=d1, y=d2, hue=conf, palette = 'tab10_r')
                    plt.legend(loc='center left', bbox_to_anchor=(1, 0.8), ncol=1, title = conf)
                else:
                    plt.figure(figsize=(8, 6))
                    plt.scatter(viz_df[d1], viz_df[d2], c=viz_df[conf], cmap = cmap_div, s=15)
                    plt.colorbar(label=conf)
                plt.title(f"{method}: {conf} (k={k})\nSil: {sil_score:.2f}")
                safe_conf = conf.replace('/', '_')
                plt.savefig(f'{base_path}_{safe_conf}.png')
                # plt.show()
                plt.close()
                print('.', end = '')

['PTAGE']
three_diags - desikan_coeffs...
Silhouette score: 0.11
PCA...
TSNE...
UMAP...


c:\Users\User\miniconda3\envs\venv_1\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...................

C:\Users\User\AppData\Local\Temp\ipykernel_10636\4000131981.py:136: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(8, 6))


.

...

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>